# ATML PA2: Task 3 - Group Relative Policy Optimization (GRPO)

This notebook runs all Task 3 experiments on a Kaggle GPU environment (T4 x 1 or P100):
1. **Environment & Assets Setup**
2. **Step 1: Standard GRPO Continuation (20 updates, K=4, truncation masking)**
3. **Step 1: Held-Out Evaluation (Reward, KL, Entropy, Length, IQR, EOS Rate)**
4. **Step 2: Equal-Generation Group-Size Study (K in {2, 4, 8} on cached rollout data)**
5. **Step 3: Length-Normalization Study (Canonical GRPO vs Dr. GRPO 8-update forks)**
6. **Results Packaging & Summary Inspection**

## 1. Verify GPU and Environment

In [ ]:
!nvidia-smi

## 2. Clone Your GitHub Repository
Ensure the latest Task 3 code is fetched from your repo.

In [ ]:
REPO_URL = "https://github.com/ahmad-sarfraz345/ATML-PA2-LLM-PostTraining.git"

import os
repo_name = REPO_URL.split("/")[-1].replace(".git", "")
if not os.path.exists(repo_name):
    !git clone $REPO_URL
    %cd $repo_name
else:
    %cd $repo_name
    !git fetch origin
    !git reset --hard origin/main
    !git pull

!git status

## 3. Install Dependencies & Validate Course Assets

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run Task 3 - Step 1: Standard GRPO Continuation (20 Updates)
Continues online RLHF from the supplied midpoint (`checkpoints/grpo_midpoint_policy`) with K=4 completions per prompt, truncation masking, and group-relative advantage normalization. Logs reward, KL, within-group reward std, uninformative-group fraction, policy loss, gradient norm, entropy, response length, peak VRAM, and wall-clock time.

In [ ]:
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard

## 5. Run Task 3 - Step 1: Standard GRPO Evaluation
Evaluates the continued GRPO policy on held-out prompts (`data/rl_prompt_pool_eval.jsonl`, 32 prompts matching instructor acceptance budget at max length 768).

In [ ]:
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard --max-prompts 32

## 6. Run Task 3 - Step 2: Equal-Generation Group-Size Study (K in {2, 4, 8})
Partitions the fixed 192 completions from `cached/grpo_k_cache.jsonl` into groups of K in {2, 4, 8}. Computes informative-group fraction, within-group reward standard deviation, advantage variance, and prompt-difficulty binning (Hard vs. Easy prompts based on median cached reward).

In [ ]:
!python -m task3_grpo.analyze_group_size --config configs/grpo.yaml

## 7. Run Task 3 - Step 3: Sequence-Length Normalization Study
Runs matched 8-update continuation forks from the midpoint checkpoint comparing Canonical GRPO (1/T_k) vs. Dr. GRPO (1/L, L=512). Evaluates held-out performance and length-conditioned gradient statistics.

In [ ]:
!python -m task3_grpo.compare_normalization --config configs/grpo.yaml --max-eval-prompts 32

## 8. View and Inspect All Generated Results

In [ ]:
import pandas as pd
import json
from pathlib import Path

res_dir = Path("results/task3_grpo")

print("=== STANDARD GRPO EVALUATION ===")
std_json = res_dir / "standard_eval.json"
if std_json.exists():
    print(json.dumps(json.loads(std_json.read_text()), indent=2))

print("\n=== GROUP-SIZE STUDY SUMMARY (K in {2, 4, 8}) ===")
k_csv = res_dir / "group_size_analysis.csv"
if k_csv.exists():
    display(pd.read_csv(k_csv))

print("\n=== NORMALIZATION STUDY SUMMARY (Canonical vs Dr. GRPO) ===")
norm_csv = res_dir / "normalization_comparison.csv"
if norm_csv.exists():
    display(pd.read_csv(norm_csv))

## 9. Package Results for Download
Zips all generated JSON/CSV results from `results/task3_grpo` so you can download them directly from Kaggle.

In [ ]:
!zip -r task3_results.zip results/task3_grpo
print("task3_results.zip ready in Kaggle working directory!")

# Optional: push results back to your GitHub repository
# !git config --global user.email "your_email@example.com"
# !git config --global user.name "Your Name"
# !git add results/task3_grpo
# !git commit -m "Add Task 3 GRPO results and logs"
# !git push